# Module 3 — Pangenomics: core, accessory, and redundancy in the SynCom
### 27221 Microbiome Engineering — proposed 4h session, direct continuation of Module 2

**How to use this document:** reference material — copy commands into your
terminal (via ThinLinc). `>> YOUR DECISION <<` marks a judgment call.

---

## 0. Context

In Module 2 you recovered genomes from a shotgun metagenome and matched them
against known SynCom isolate genomes. Today's question is different: instead
of asking *"what's present, and how much of it?"*, pangenomics asks
*"how much gene content do these isolates share, and how much is unique to
each?"*

This is a genuinely engineering-relevant question for a **designed** community.
If two SynCom members share almost all of their gene content, they may be
functionally redundant — dropping one might not change what the community can
do. If a member carries a large, unique accessory genome, it may be
contributing something no other member can replace. Either answer is useful
information for someone deciding whether a 15-member SynCom could be
simplified to 8 without losing function.

anvi'o builds a pangenome by translating every gene in every input genome into
protein sequence, all-against-all comparing them, and clustering similar
sequences into **gene clusters** — a gene cluster present in every genome is
part of the **core** genome; one present in only some genomes is **accessory**;
one present in only a single genome is a **singleton**.

---

## 1. Setup

In [ ]:
conda activate anvio-8

# --- SESSION CONFIG: adjust these paths if your instructor gives you different ones ---
export WORKDIR=~/module3_output
export ISOLATE_GENOMES_DIR=/path/to/shared/syncom_metagenomics/isolate_genomes   # same genomes as Module 2
export EXTERNAL_GENOMES_FILE=/path/to/shared/syncom_metagenomics/external-genomes.txt  # same file as Module 2, Step 5
mkdir -p $WORKDIR
cd $WORKDIR

In [ ]:
# Sanity check: how many isolate genomes are we working with?
cat $EXTERNAL_GENOMES_FILE

**Question:** How many isolate genomes are in your SynCom? Do you remember
roughly how phylogenetically related they are to each other (same genus?
same family? more distant?) — that will shape what you expect to see in the
core/accessory split today.

---

## 2. Build a genomes storage

The **genomes storage** is anvi'o's container for multiple genomes' worth of
gene calls and annotations, analogous to what a single contigs database holds
for one assembly.

In [ ]:
anvi-gen-genomes-storage \
  -e $EXTERNAL_GENOMES_FILE \
  -o SYNCOM-GENOMES.db

*(If you'd like to include the MAGs you recovered in Module 2 alongside the
isolates, your instructor can show you how to add an `internal-genomes.txt`
file here with the `-i` flag — this lets you see where your recovered genomes
land relative to the known isolates in gene-content space.)*

---

## 3. Run the pangenome analysis

In [ ]:
anvi-pan-genome \
  -g SYNCOM-GENOMES.db \
  -n SynCom_Pan \
  --output-dir $WORKDIR/SynCom_Pan \
  --num-threads 4 \
  --minbit 0.5 \
  --mcl-inflation 10

**`>> YOUR DECISION <<`** — `--mcl-inflation` controls how "strict" protein
clustering is: higher values produce more, tighter gene clusters (better for
closely related genomes where you want to resolve fine differences); lower
values merge more permissively (often used for more distantly related
genomes). We've suggested 10, a common default for genomes within the same
genus. Given what you noted about your SynCom's relatedness in Step 1, does
this seem like the right choice, or would you consider adjusting it?

---

## 4. Explore interactively

In [ ]:
anvi-display-pan -g SYNCOM-GENOMES.db -p $WORKDIR/SynCom_Pan/SynCom_Pan-PAN.db

Open the printed URL in a browser tab inside your ThinLinc session. Each
"spoke" is a gene cluster; genomes are arranged as rings showing presence/
absence of each cluster.

**Question:** Visually, does one genome look like an outlier (many
unique-looking gene clusters), or do they all look broadly similar? Does this
match your expectation from Step 1?

---

## 5. Extract core/accessory/singleton counts

In [ ]:
anvi-summarize \
  -g SYNCOM-GENOMES.db \
  -p $WORKDIR/SynCom_Pan/SynCom_Pan-PAN.db \
  -C default \
  -o $WORKDIR/pan_summary

Open `$WORKDIR/pan_summary/SynCom_Pan_gene_clusters_summary.txt` — this table
tells you, per gene cluster, which genomes it's present in.

**Question:** Roughly what fraction of all gene clusters are core (present in
every genome) vs. accessory (present in some) vs. singleton (present in only
one)? For each isolate, which one has the largest fraction of singleton gene
clusters — i.e., which looks most functionally distinct from the rest of the
SynCom?

---

## 6. Functional enrichment of the accessory genome

This step reuses the COG functional annotation from Module 2 (Step 5,
`anvi-run-ncbi-cogs`) — if that wasn't run on these particular isolate
genomes yet, your instructor will point you to where it's staged.

In [ ]:
anvi-compute-functional-enrichment-in-pan \
  -p $WORKDIR/SynCom_Pan/SynCom_Pan-PAN.db \
  -g SYNCOM-GENOMES.db \
  --category-variable source \
  --annotation-source COG20_FUNCTION \
  -o $WORKDIR/functional_enrichment.txt

**Question:** Are there functional categories enriched in the accessory
genome of a particular isolate — anything that stands out as a distinct
capability that isolate might be contributing to the community?

---

## 7. Produce your deliverable

**Choose ONE figure**: an annotated `anvi-display-pan` screenshot, or a simple
bar chart of core/accessory/singleton gene cluster counts per isolate
(built from the summary table in Step 5).

**Write your interpretation (2–5 sentences):** which isolate(s) look most
functionally redundant with the rest of the SynCom, which look most distinct,
and what would you recommend if someone asked whether this SynCom could be
simplified without losing function?

---

## Wrap-up discussion

- How does today's gene-content view compare to the amplicon-based (Module 1)
  and genome-recovery-based (Module 2) views of the same kind of system? What
  does pangenomics tell you that neither of the others could?
- If two isolates turned out to share nearly all of their gene content, would
  that alone be enough to recommend dropping one from the SynCom — what
  additional evidence (from the actual plant experiment) would you want
  before making that call?